# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [19]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests

# Import API key
from api_keys import geoapify_key

In [20]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,alakurtti,66.9672,30.3491,-0.01,89,29,3.17,RU,1743627027
1,1,college,64.8569,-147.8028,1.76,87,100,3.60,US,1743627028
2,2,nemuro,43.3236,145.5750,0.28,95,100,12.15,JP,1743627030
3,3,port elizabeth,-33.9180,25.5701,15.73,94,0,2.06,ZA,1743627031
4,4,grytviken,-54.2811,-36.5092,5.28,98,100,9.66,GS,1743627032


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [21]:
# Configure the map plot
city_map = city_data_df.hvplot.points(
    "Lng",
    "Lat",
    geo=True,
    tiles="OSM",
    size="Humidity",
    color="City",
    alpha=0.5,
    hover_cols=["City", "Country", "Humidity"]
)

# Display the map
city_map

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Country)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [22]:
# Narrow down cities that fit criteria and drop any results with null values
ideal_cities_df = city_data_df.loc[
    (city_data_df["Max Temp"] < 27) &
    (city_data_df["Max Temp"] > 21) &
    (city_data_df["Wind Speed"] < 4.5) &
    (city_data_df["Cloudiness"] == 0)
]
# Drop any rows with null values
ideal_cities_df = ideal_cities_df.dropna()

# Display sample data
ideal_cities_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
26,26,veraval,20.9000,70.3667,25.85,60,0,4.23,IN,1743627059
100,100,antonio enes,-16.2325,39.9086,26.24,83,0,1.94,MZ,1743627151
112,112,holualoa,19.6228,-155.9522,26.97,58,0,4.12,US,1743627166
121,121,crespo,-32.0287,-60.3066,24.46,37,0,1.73,AR,1743626682
173,173,saint-pierre,-21.3393,55.4781,23.82,83,0,2.57,RE,1743627241


### Step 3: Create a new DataFrame called `hotel_df`.

In [23]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity
hotel_df = ideal_cities_df[["City", "Country", "Lat", "Lng", "Humidity"]].copy()

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df["Hotel Name"] = ""

# Display sample data
hotel_df.head()

,City,Country,Lat,Lng,Humidity,Hotel Name
26,veraval,IN,20.9000,70.3667,60,
100,antonio enes,MZ,-16.2325,39.9086,83,
112,holualoa,US,19.6228,-155.9522,58,
121,crespo,AR,-32.0287,-60.3066,37,
173,saint-pierre,RE,-21.3393,55.4781,83,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [24]:
# Set parameters to search for a hotel
radius = 10000
params = {
    "categories": "accommodation.hotel",
    "limit": 1,
    "bias": "proximity",
    "apiKey": geoapify_key,
    "filter": f"circle:{hotel_df['Lng']},{hotel_df['Lat']},{radius}"
}

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    lat = row["Lat"]
    lng = row["Lng"]

    # Add the current city's latitude and longitude to the params dictionary
    params["filter"] = f"circle:{lng},{lat},{radius}"
    params["bias"] = f"proximity:{lng},{lat}"

    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"

    # Make and API request using the params dictionary
    name_address = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    name_address = name_address.json()

    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Starting hotel search
veraval - nearest hotel: Shyam
antonio enes - nearest hotel: Hotel Quirimbas
holualoa - nearest hotel: Kona Hotel
crespo - nearest hotel: Hotel Garten
saint-pierre - nearest hotel: Tropic Hotel
sawai madhopur - nearest hotel: Ranthambore Tiger Valley
idri - nearest hotel: No hotel found
tamanrasset - nearest hotel: فندق الأمان
tindouf - nearest hotel: محمد بوسبي
akhmim - nearest hotel: فندق نايل تريجر سوهاج
rapar - nearest hotel: No hotel found
brak - nearest hotel: فندق براك السياحي
tsiombe - nearest hotel: No hotel found
adh dhayd - nearest hotel: No hotel found


,City,Country,Lat,Lng,Humidity,Hotel Name
26,veraval,IN,20.9000,70.3667,60,Shyam
100,antonio enes,MZ,-16.2325,39.9086,83,Hotel Quirimbas
112,holualoa,US,19.6228,-155.9522,58,Kona Hotel
121,crespo,AR,-32.0287,-60.3066,37,Hotel Garten
173,saint-pierre,RE,-21.3393,55.4781,83,Tropic Hotel
251,sawai madhopur,IN,25.9833,76.3667,10,Ranthambore Tiger Valley
283,idri,LY,27.5000,13.2667,12,No hotel found
324,tamanrasset,DZ,22.7850,5.5228,14,فندق الأمان
343,tindouf,DZ,27.6711,-8.1474,33,محمد بوسبي
357,akhmim,EG,26.5622,31.7450,22,فندق نايل تريجر سوهاج


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [26]:
# Configure the map plot
hotel_map = hotel_df.hvplot.points(
    "Lng",
    "Lat",
    geo=True,
    tiles="OSM",
    size="Humidity",
    color="Hotel Name",
    alpha=0.5,
    hover_cols=["City", "Country", "Hotel Name"]
)
hotel_map = hotel_map.opts(
    title="Hotels in Ideal Cities",
    frame_width=800,
    frame_height=400
)
# Display the map
hotel_map

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (Hotel Name,Humidity,City,Country)